In [1]:
# import os
# import numpy as np
# import pandas as pd
# import torchaudio
# import librosa
# from scipy.signal import correlate

# # ---------------------------- Feature Extraction Helpers ---------------------------- #

# # def vad_from_rms(rms, threshold=0.02):
# #     return (rms > threshold).astype(int)

# # def speaking_time(activity, frame_hop):
# #     return np.sum(activity) * frame_hop

# def pause_stats_from_activity(activity, frame_hop):
#     pauses, current_pause = [], 0
#     for a in activity:
#         if a == 0:
#             current_pause += 1
#         elif current_pause > 0:
#             pauses.append(current_pause)
#             current_pause = 0
#     if current_pause > 0:
#         pauses.append(current_pause)
#     pauses = np.array(pauses) * frame_hop if len(pauses) else np.array([0])
#     total_pause = np.sum(pauses)
#     avg_pause = np.mean(pauses) if len(pauses) else 0
#     long_pause_count = np.sum(pauses > 10.0)
#     short_pause_count = np.sum((pauses >= 3.0) & (pauses < 5.0))
#     silence_ratio = total_pause / (len(activity) * frame_hop)
#     return total_pause, avg_pause, long_pause_count, short_pause_count, silence_ratio

# # def overlap_and_turns(act_a, act_b):
# #     overlap = np.sum((act_a == 1) & (act_b == 1))
# #     turns = np.sum(np.diff(np.sign(act_a - act_b)) != 0)
# #     return overlap, turns

# def pitch_contour(y, sr):
#     f0, _, _ = librosa.pyin(y, fmin=80, fmax=400, sr=sr)
#     f0 = f0[~np.isnan(f0)]
#     return (np.mean(f0), np.std(f0)) if len(f0) else (0, 0)

# # def zero_crossing_rate(y):
# #     return np.mean(librosa.feature.zero_crossing_rate(y)[0])

# # def align_with_xcorr(a_rms, b_rms, sr_hop):
# #     corr = correlate(a_rms - np.mean(a_rms), b_rms - np.mean(b_rms), mode='full')
# #     lag = np.argmax(corr) - (len(a_rms) - 1)
# #     return lag * sr_hop

# # ---------------------------- Main Processing ---------------------------- #

# def extract_features_from_pair(path_a, path_b, frame_len=0.025, frame_hop=0.010):
#     yA, srA = torchaudio.load(path_a)
#     yB, srB = torchaudio.load(path_b)
#     yA, yB = yA[0].numpy(), yB[0].numpy()
#     assert srA == srB, "Sampling rates differ!"
#     sr = srA

#     # rmsA = librosa.feature.rms(y=yA, frame_length=int(sr*frame_len), hop_length=int(sr*frame_hop))[0]
#     # rmsB = librosa.feature.rms(y=yB, frame_length=int(sr*frame_len), hop_length=int(sr*frame_hop))[0]
#     # actA, actB = vad_from_rms(rmsA), vad_from_rms(rmsB)

#     # speakA, speakB = speaking_time(actA, frame_hop), speaking_time(actB, frame_hop)
#     # ratioA, ratioB = speakA / (speakA + speakB + 1e-6), speakB / (speakA + speakB + 1e-6)

#     total_pauseA, avg_pauseA, longA, shortA, silenceA = pause_stats_from_activity(actA, frame_hop)
#     total_pauseB, avg_pauseB, longB, shortB, silenceB = pause_stats_from_activity(actB, frame_hop)

#     overlap, switches = overlap_and_turns(actA, actB)
#     overlap_time = overlap * frame_hop

#     meanF0_A, stdF0_A = pitch_contour(yA, sr)
#     meanF0_B, stdF0_B = pitch_contour(yB, sr)

#     # zcrA, zcrB = zero_crossing_rate(yA), zero_crossing_rate(yB)
#     # align_lag = align_with_xcorr(rmsA, rmsB, frame_hop)

#     energy_std_A, energy_std_B = np.std(rmsA), np.std(rmsB)

#     # Speaker-specific and shared features
#     features_A = {
#         "speaker_id": os.path.basename(path_a),
#         "speak_time_s": speakA,
#         "speak_ratio": ratioA,
#         "pause_total_s": total_pauseA,
#         "pause_avg_s": avg_pauseA,
#         "pause_short_count": shortA,
#         "pause_long_count": longA,
#         "silence_ratio": silenceA,
#         "meanF0_Hz": meanF0_A,
#         "stdF0_Hz": stdF0_A,
#         "ZCR": zcrA,
#         "energy_std": energy_std_A,
#         "align_lag_s": align_lag,
#         "overlap_s": overlap_time,
#         "floor_switches": switches,
#     }

#     features_B = {
#         "speaker_id": os.path.basename(path_b),
#         "speak_time_s": speakB,
#         "speak_ratio": ratioB,
#         "pause_total_s": total_pauseB,
#         "pause_avg_s": avg_pauseB,
#         "pause_short_count": shortB,
#         "pause_long_count": longB,
#         "silence_ratio": silenceB,
#         "meanF0_Hz": meanF0_B,
#         "stdF0_Hz": stdF0_B,
#         "ZCR": zcrB,
#         "energy_std": energy_std_B,
#         "align_lag_s": align_lag,
#         "overlap_s": overlap_time,
#         "floor_switches": switches,
#     }

#     return features_A, features_B


# # ---------------------------- Run for All Pairs ---------------------------- #

# # Example: [(P01_1, P01_2), (P02_1, P02_2), ...]
# pairs = [(f"/kaggle/input/audio-files/P{i:02d}_1.wav", f"/kaggle/input/audio-files/P{i:02d}_2.wav") for i in range(1, 3)]  # adjust range as needed

# for path_a, path_b in pairs:
#     if not (os.path.exists(path_a) and os.path.exists(path_b)):
#         print(f"Skipping {path_a}, {path_b} — file missing.")
#         continue

#     feats_A, feats_B = extract_features_from_pair(path_a, path_b)

#     # Save CSV per speaker
#     pd.DataFrame([feats_A]).to_csv(f"{os.path.splitext(path_a)[0]}_features.csv", index=False)
#     pd.DataFrame([feats_B]).to_csv(f"{os.path.splitext(path_b)[0]}_features.csv", index=False)

#     print(f" Saved features for {path_a} and {path_b}")


ValueError: operands could not be broadcast together with shapes (65200,) (65039,) 

In [12]:
pip install torch torchaudio librosa pandas


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 70.3 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 61.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 36.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 5.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 3.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 2.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 24.3 MB/s eta 0:00:0000:0100:01
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.

In [24]:
# import os
# import torch
# import pandas as pd

# # ---------------- LOAD SILERO VAD ----------------
# model, utils = torch.hub.load('snakers4/silero-vad', model='silero_vad')
# (get_speech_timestamps, save_audio, read_audio, VADIterator, collect_chunks) = utils

# # ---------------- CONFIG ----------------
# audio_folder = "/kaggle/input/audio-files"
# output_folder = "pairwise_csv_outputs"
# os.makedirs(output_folder, exist_ok=True)
# SAMPLING_RATE = 16000

# # ---------------- STEP 1: GET VAD SEGMENTS ----------------
# def get_vad_segments(file_path):
#     """Run Silero VAD and return list of (start, end) in seconds."""
#     wav = read_audio(file_path, sampling_rate=SAMPLING_RATE)
#     timestamps = get_speech_timestamps(wav, model, sampling_rate=SAMPLING_RATE)
#     return [(ts['start']/SAMPLING_RATE, ts['end']/SAMPLING_RATE) for ts in timestamps]

# # ---------------- STEP 2: COMPUTE INTERACTION FEATURES ----------------
# def compute_pair_features(vadA, vadB):
#     """Compute overlaps, turn-taking (A→B/B→A), gaps, and speech dominance."""
#     total_speech_A = sum(e - s for s, e in vadA)
#     total_speech_B = sum(e - s for s, e in vadB)

#     # ---- Overlap ----
#     overlap_count = 0
#     overlap_duration = 0
#     for sA, eA in vadA:
#         for sB, eB in vadB:
#             inter = max(0, min(eA, eB) - max(sA, sB))
#             if inter > 0:
#                 overlap_duration += inter
#                 overlap_count += 1

#     # ---- Turn-taking ----
#     events = [(s, e, "A") for s, e in vadA] + [(s, e, "B") for s, e in vadB]
#     events.sort(key=lambda x: x[0])

#     turn_AB = 0
#     turn_BA = 0
#     gaps = []  # collect gap durations between turns

#     for i in range(1, len(events)):
#         prev = events[i - 1]
#         curr = events[i]

#         # If non-overlapping transition
#         if curr[0] >= prev[1]:
#             gap = curr[0] - prev[1]
#             gaps.append(gap)

#             if prev[2] == "A" and curr[2] == "B":
#                 turn_AB += 1
#             elif prev[2] == "B" and curr[2] == "A":
#                 turn_BA += 1

#     total_turns = turn_AB + turn_BA
#     avg_gap = sum(gaps) / len(gaps) if gaps else 0

#     # ---- Speech Dominance ----
#     total_speech = total_speech_A + total_speech_B
#     dominance_A = total_speech_A / (total_speech + 1e-6)
#     dominance_B = total_speech_B / (total_speech + 1e-6)

#     return {
#         "total_speech_A_s": total_speech_A,
#         "total_speech_B_s": total_speech_B,
#         "overlap_count": overlap_count,
#         "overlap_duration_s": overlap_duration,
#         "turns_A_to_B": turn_AB,
#         "turns_B_to_A": turn_BA,
#         "total_turns": total_turns,
#         "avg_gap_s": avg_gap,
#         "dominance_A": dominance_A, #range 0 to 1 so if A> 0.5 than A is dominant
#         "dominance_B": dominance_B,
#     }

# # ---------------- STEP 3: MAIN LOOP ----------------
# pairs = [
#     ("P01_1.wav", "P01_2.wav"),
#     ("P02_1.wav", "P02_2.wav"),
#     # add more pairs here
# ]

# results = []

# for spkA, spkB in pairs:
#     pathA = os.path.join(audio_folder, spkA)
#     pathB = os.path.join(audio_folder, spkB)
#     print(f"Processing pair: {spkA} & {spkB}")

#     vadA = get_vad_segments(pathA)
#     vadB = get_vad_segments(pathB)

#     pair_feats = compute_pair_features(vadA, vadB)
#     pair_feats["pair_id"] = f"{spkA}_{spkB}"
#     results.append(pair_feats)

# # ---------------- SAVE SUMMARY ----------------
# summary_df = pd.DataFrame(results)
# summary_df.to_csv(os.path.join(output_folder, "pairwise_summary.csv"), index=False)

# print("✅ Done! Summary CSV saved in:", output_folder)


Using cache found in /root/.cache/torch/hub/snakers4_silero-vad_master


Processing pair: P01_1.wav & P01_2.wav
Processing pair: P02_1.wav & P02_2.wav
✅ Done! Summary CSV saved in: pairwise_csv_outputs


In [40]:
import os
import torch
import pandas as pd
import torchaudio
import numpy as np

# ---------------- LOAD SILERO VAD ----------------
model, utils = torch.hub.load('snakers4/silero-vad', model='silero_vad')
(get_speech_timestamps, save_audio, read_audio, VADIterator, collect_chunks) = utils

# ---------------- CONFIG ----------------
audio_folder = "/kaggle/input/audio-files"
output_folder = "pairwise_csv_outputs"
os.makedirs(output_folder, exist_ok=True)
SAMPLING_RATE = 16000

SHORT_PAUSE_THRESHOLD = 5.0  # sec
LONG_PAUSE_THRESHOLD = 10.0   # sec

# ---------------- STEP 1: GET VAD SEGMENTS ----------------
def get_vad_segments(file_path):
    """Run Silero VAD and return list of (start, end) in seconds."""
    wav = read_audio(file_path, sampling_rate=SAMPLING_RATE)
    timestamps = get_speech_timestamps(wav, model, sampling_rate=SAMPLING_RATE)
    return [(ts['start'] / SAMPLING_RATE, ts['end'] / SAMPLING_RATE) for ts in timestamps]

# ---------------- STEP 2: COMPUTE PAUSE FEATURES ----------------
def compute_pause_features(vad_segments):
    """Compute pause-related stats for one speaker."""
    pauses = []
    for i in range(1, len(vad_segments)):
        prev_end = vad_segments[i - 1][1]
        curr_start = vad_segments[i][0]
        pause_dur = curr_start - prev_end
        if pause_dur > 0:
            pauses.append(pause_dur)

    total_pauses = len(pauses)
    short_pauses = [p for p in pauses if p < SHORT_PAUSE_THRESHOLD]
    long_pauses = [p for p in pauses if p >= LONG_PAUSE_THRESHOLD]

    return {
        "total_pauses": total_pauses,
        "short_pauses": len(short_pauses),
        "long_pauses": len(long_pauses),
        "avg_pause_dur_s": sum(pauses)/len(pauses) if pauses else 0,
        "total_silence_s": sum(pauses),
        "long_pause_dur_s": sum(long_pauses)
    }

# ---------------- STEP 3: COMPUTE INTERACTION FEATURES ----------------
def compute_pair_features(vadA, vadB):
    """Compute pair-level conversational interaction metrics."""
    total_speech_A = sum(e - s for s, e in vadA)
    total_speech_B = sum(e - s for s, e in vadB)

    # ---- Overlap ----
    overlap_count = 0
    overlap_duration = 0
    for sA, eA in vadA:
        for sB, eB in vadB:
            inter = max(0, min(eA, eB) - max(sA, sB))
            if inter > 0:
                overlap_duration += inter
                overlap_count += 1

    # ---- Turn-taking ----
    events = [(s, e, "A") for s, e in vadA] + [(s, e, "B") for s, e in vadB]
    events.sort(key=lambda x: x[0])

    turn_AB = 0
    turn_BA = 0
    gaps = []

    for i in range(1, len(events)):
        prev = events[i - 1]
        curr = events[i]
        if curr[0] >= prev[1]:
            gap = curr[0] - prev[1]
            gaps.append(gap)
            if prev[2] == "A" and curr[2] == "B":
                turn_AB += 1
            elif prev[2] == "B" and curr[2] == "A":
                turn_BA += 1

    total_turns = turn_AB + turn_BA
    avg_gap = sum(gaps) / len(gaps) if gaps else 0

    # ---- Speech Dominance ----
    total_speech = total_speech_A + total_speech_B
    dominance_A = total_speech_A / (total_speech + 1e-6)
    dominance_B = total_speech_B / (total_speech + 1e-6)

    return {
        "total_speech_A_s": total_speech_A,
        "total_speech_B_s": total_speech_B,
        "overlap_count": overlap_count,
        "overlap_duration_s": overlap_duration,
        "turns_A_to_B": turn_AB,
        "turns_B_to_A": turn_BA,
        "total_turns": total_turns,
        "avg_gap_s": avg_gap,
        "dominance_A": dominance_A,
        "dominance_B": dominance_B
    }

# ---------------- STEP 4: PITCH & ENERGY ----------------
import librosa
import numpy as np

def compute_pitch_energy(file_path, vad_segments):
    y, sr = librosa.load(file_path, sr=None)
    all_pitch = []
    all_energy = []

    for seg in vad_segments:
        if isinstance(seg, dict):
            start, end = seg["start"], seg["end"]
        else:
            start, end = seg  # tuple or list

        start_sample = int(start * sr)
        end_sample = int(end * sr)
        segment = y[start_sample:end_sample]

        # Skip too-short segments
        if len(segment) < 0.05 * sr:
            continue

        # ---- Compute pitch using PYIN ----
        try:
            pitches, _, _ = librosa.pyin(segment, 
                                         fmin=50, fmax=500, 
                                         sr=sr, frame_length=1024)
            valid_pitch = pitches[~np.isnan(pitches)]
            if len(valid_pitch) > 0:
                all_pitch.append(np.mean(valid_pitch))
        except Exception as e:
            print(f"Pitch error: {e}")

        # ---- Compute energy ----
        energy = np.sum(segment**2) / len(segment)
        all_energy.append(energy)

    if len(all_pitch) == 0:
        avg_pitch = min_pitch = max_pitch = 0
    else:
        avg_pitch = float(np.mean(all_pitch))
        min_pitch = float(np.min(all_pitch))
        max_pitch = float(np.max(all_pitch))

    if len(all_energy) == 0:
        avg_energy = min_energy = max_energy = 0
    else:
        avg_energy = float(np.mean(all_energy))
        min_energy = float(np.min(all_energy))
        max_energy = float(np.max(all_energy))

    return {
        "avg_pitch_Hz": avg_pitch,
        "min_pitch_Hz": min_pitch,
        "max_pitch_Hz": max_pitch,
        "avg_energy": avg_energy,
        "min_energy": min_energy,
        "max_energy": max_energy,
    }




# ---------------- STEP 5: MAIN LOOP ----------------
pairs = [
    ("P01_1.wav", "P01_2.wav"),
    ("P02_1.wav", "P02_2.wav"),
    # add more pairs here
]

pairwise_results = []
individual_results = []
timestamp_records = []  # to store start-end for each speaker

for spkA, spkB in pairs:
    pathA = os.path.join(audio_folder, spkA)
    pathB = os.path.join(audio_folder, spkB)
    print(f"Processing pair: {spkA} & {spkB}")

    vadA = get_vad_segments(pathA)
    vadB = get_vad_segments(pathB)

    # --- Save timestamps for reference ---
    for s, e in vadA:
        timestamp_records.append({"pair_id": f"{spkA}_{spkB}", "speaker_id": spkA, "start_s": s, "end_s": e})
    for s, e in vadB:
        timestamp_records.append({"pair_id": f"{spkA}_{spkB}", "speaker_id": spkB, "start_s": s, "end_s": e})

    # --- Pairwise (common) features ---
    pair_feats = compute_pair_features(vadA, vadB)
    pair_feats["pair_id"] = f"{spkA}_{spkB}"
    pairwise_results.append(pair_feats)

    # --- Individual features (A and B separately) ---
    pauseA = compute_pause_features(vadA)
    pauseB = compute_pause_features(vadB)

    pitchA = compute_pitch_energy(pathA, vadA)
    pitchB = compute_pitch_energy(pathB, vadB)

    individual_results.append({
        "pair_id": f"{spkA}_{spkB}",
        "speaker_id": spkA,
        **pauseA,
        **pitchA,
        "total_speech_s": pair_feats["total_speech_A_s"],
        "dominance": pair_feats["dominance_A"]
    })

    individual_results.append({
        "pair_id": f"{spkA}_{spkB}",
        "speaker_id": spkB,
        **pauseB,
        **pitchB,
        "total_speech_s": pair_feats["total_speech_B_s"],
        "dominance": pair_feats["dominance_B"]
    })

# ---------------- STEP 6: SAVE SEPARATE SUMMARIES ----------------
pairwise_df = pd.DataFrame(pairwise_results)
individual_df = pd.DataFrame(individual_results)
timestamps_df = pd.DataFrame(timestamp_records)

pairwise_path = os.path.join(output_folder, "pairwise_summary.csv")
individual_path = os.path.join(output_folder, "individual_summary.csv")
timestamps_path = os.path.join(output_folder, "timestamps_summary.csv")

pairwise_df.to_csv(pairwise_path, index=False)
individual_df.to_csv(individual_path, index=False)
timestamps_df.to_csv(timestamps_path, index=False)

print("✅ Done!")
print("Pairwise summary saved at:", pairwise_path)
print("Individual summary saved at:", individual_path)
print("Timestamps saved at:", timestamps_path)


Using cache found in /root/.cache/torch/hub/snakers4_silero-vad_master


Processing pair: P01_1.wav & P01_2.wav
Processing pair: P02_1.wav & P02_2.wav
✅ Done!
Pairwise summary saved at: pairwise_csv_outputs/pairwise_summary.csv
Individual summary saved at: pairwise_csv_outputs/individual_summary.csv
Timestamps saved at: pairwise_csv_outputs/timestamps_summary.csv


In [36]:
file="/kaggle/working/pairwise_csv_outputs/pairwise_summary.csv"
df=pd.read_csv(file)
print(df)

   total_speech_A_s  total_speech_B_s  overlap_count  overlap_duration_s  \
0            95.332            69.876             45              24.300   
1            86.948           141.152             64              26.336   

   turns_A_to_B  turns_B_to_A  total_turns  avg_gap_s  dominance_A  \
0            28            42           70   3.827866     0.577042   
1            50            54          104   1.856646     0.381184   

   dominance_B              pair_id  
0     0.422958  P01_1.wav_P01_2.wav  
1     0.618816  P02_1.wav_P02_2.wav  


In [41]:
file="/kaggle/working/pairwise_csv_outputs/individual_summary.csv"
df=pd.read_csv(file)
print(df)

               pair_id speaker_id  total_pauses  short_pauses  long_pauses  \
0  P01_1.wav_P01_2.wav  P01_1.wav            78            54           18   
1  P01_1.wav_P01_2.wav  P01_2.wav            82            53           18   
2  P02_1.wav_P02_2.wav  P02_1.wav           110            73           14   
3  P02_1.wav_P02_2.wav  P02_2.wav           175           146            6   

   avg_pause_dur_s  total_silence_s  long_pause_dur_s  avg_pitch_Hz  \
0         6.234154          486.264           337.160    220.773296   
1         6.348976          520.616           352.008    258.651660   
2         4.799055          527.896           255.000    171.993454   
3         2.666949          466.716           108.792    167.398798   

   min_pitch_Hz  max_pitch_Hz  avg_energy  min_energy  max_energy  \
0    168.774481    348.080619    0.000166    0.000001    0.001077   
1    175.139188    424.753392    0.011720    0.000226    0.080228   
2    101.793858    463.510451    0.000794    0

In [42]:
file="/kaggle/working/pairwise_csv_outputs/P01_1_VAD.csv"
df=pd.read_csv(file)
print(df)

    start_time_s  end_time_s speaker              pair_id  overlap_s  \
0          8.450       8.926       A  P01_1.wav_P01_2.wav       24.3   
1         32.738      34.366       A  P01_1.wav_P01_2.wav       24.3   
2         44.994      45.438       A  P01_1.wav_P01_2.wav       24.3   
3         46.082      47.262       A  P01_1.wav_P01_2.wav       24.3   
4         62.658      65.758       A  P01_1.wav_P01_2.wav       24.3   
..           ...         ...     ...                  ...        ...   
74       570.338     570.718       A  P01_1.wav_P01_2.wav       24.3   
75       575.426     576.094       A  P01_1.wav_P01_2.wav       24.3   
76       583.394     584.126       A  P01_1.wav_P01_2.wav       24.3   
77       585.090     586.142       A  P01_1.wav_P01_2.wav       24.3   
78       589.634     590.046       A  P01_1.wav_P01_2.wav       24.3   

    turn_takings  speech_ratio  
0            112      0.577042  
1            112      0.577042  
2            112      0.577042  
3  